## Setup

In [ ]:
source("~/workspace/pipelines/snt_dhis2_formatting/utils/snt_dhis2_formatting.r")
snt_paths <- init_snt_workspace(
    snt_pipeline_name="snt_dhis2_formatting",
    packages=c("lubridate", "arrow", "dplyr", "stringi", "stringr", "jsonlite", "httr", "glue"))
    
# Load config
config_json <- load_snt_config(file.path(snt_paths$CONFIG_PATH, "SNT_config.json"))

# Save config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)
extracts_dataset_id <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_EXTRACTS

### Load DHIS2 pyramid data  

-Load DHIS2 pyramid from latest dataset version

In [ ]:
# Load file from dataset
dhis2_pyramid_data <- load_dataset_file(extracts_dataset_id, paste0(COUNTRY_CODE, "_dhis2_raw_pyramid.parquet"), verbose=FALSE)
log_msg(glue("DHIS2 organisation units data loaded from dataset : '{extracts_dataset_id}' dataframe dimensions: {paste(dim(dhis2_pyramid_data), collapse=', ')}"))
head(dhis2_pyramid_data, 3)

## SNT pyramid formatting

In [ ]:
log_msg(glue("Start DHIS2 organisation units(pyramid) formatting."))   

# Standard string formats 
pyramid_data <- clean_input_data(dhis2_pyramid_data, verbose=FALSE)
head(pyramid_data, 3)

### Extract longitude/latitude from geometry column (geoJson)

In [ ]:
# Extract lon/lat from geometry
pyramid_data_coords <- extract_geometry_coordinates(pyramid_data, geom_col = "GEOMETRY")
head(pyramid_data_coords, 3)

### Try coordinates validation steps

In [ ]:
# Step 1 - Try Load country border from folder (if exists)
shapes_sf <- read_geojson_safe(file.path("~/workspace/data/dhis2/extracts_formatted/" , paste0(COUNTRY_CODE, "_shapes.geojson")))

In [ ]:
# Step 2 - Keep original coordinates already inside the country (If shapes are available)
if (!is.null(shapes_sf)) {
    log_msg("Running coordinate boundary validation")
    
    shapes_sf_boundary <- prepare_country_boundary(shapes_sf)
    lon0 <- pyramid_data_coords$LONGITUDE
    lat0 <- pyramid_data_coords$LATITUDE
    within_original <- points_within_country_batch(lon0, lat0, shapes_sf_boundary)
    has_coords <- !is.na(lon0) & !is.na(lat0)
    
    coord_fix_df <- tibble(
        LONGITUDE_ORIGINAL = lon0,
        LATITUDE_ORIGINAL = lat0,
        LONGITUDE_FIXED = NA_real_,
        LATITUDE_FIXED = NA_real_,
        COORD_FIX_METHOD = NA_character_,
        COORD_IS_VALID = FALSE
    )
    
    ok_original <- has_coords & within_original            
    if (any(ok_original)) {
        coord_fix_df$LONGITUDE_FIXED[ok_original] <- lon0[ok_original]
        coord_fix_df$LATITUDE_FIXED[ok_original] <- lat0[ok_original]
        coord_fix_df$COORD_FIX_METHOD[ok_original] <- "ORIGINAL"
        coord_fix_df$COORD_IS_VALID[ok_original] <- TRUE
    }
} else {
    log_msg("Skipped coordinate boundary validation: No reference shapes available.")
}

In [ ]:
# Step 3 - For remaining points, try correction sequence
if (!is.null(shapes_sf)) {
    
    miss <- !has_coords
    if (any(miss)) {
        coord_fix_df$COORD_FIX_METHOD[miss] <- "MISSING_COORDINATES"
    }

    need_fix <- !within_original & has_coords 
    log_msg(glue("Found {sum(need_fix)} / {length(need_fix)} coordinates that require fixing."))
    
    if (any(need_fix)) {
        idx_fix <- which(need_fix)
        fix_results <- lapply(idx_fix, function(i) {
            fix_coordinate_pair_in_country(lon0[i], lat0[i], shapes_sf_boundary, max_shift = 2)
        })
        
    fixed_coords <- sum(sapply(fix_results, function(x) x$VALID == TRUE), na.rm = TRUE)    
    log_msg(glue("Points corrected: {fixed_coords} out of {sum(need_fix)}"))
                              
    for (k in seq_along(idx_fix)) {
        i <- idx_fix[k]
        fr <- fix_results[[k]]
        coord_fix_df$LONGITUDE_FIXED[i] <- fr$LONGITUDE
        coord_fix_df$LATITUDE_FIXED[i] <- fr$LATITUDE
        coord_fix_df$COORD_FIX_METHOD[i] <- fr$METHOD
        coord_fix_df$COORD_IS_VALID[i] <- fr$VALID
      }
    }
}

In [ ]:
# Display fixed points (if any)
if (exists("fixed_coords") && length(fixed_coords) > 0) {
    my_map <- plot_fixed_coordinates(fix_results, shapes_sf_boundary)    
} 

In [ ]:
# Step 6 - Apply final coordinates and flag invalids
if (!is.null(shapes_sf)) {
    pyramid_data_coords$LONGITUDE <- coord_fix_df$LONGITUDE_FIXED
    pyramid_data_coords$LATITUDE <- coord_fix_df$LATITUDE_FIXED
    
    invalid_coords <- pyramid_data_coords %>%
        bind_cols(coord_fix_df %>% select(LONGITUDE_ORIGINAL, LATITUDE_ORIGINAL, COORD_FIX_METHOD, COORD_IS_VALID)) %>%
        filter(!COORD_IS_VALID & !is.na(LONGITUDE_ORIGINAL) & !is.na(LATITUDE_ORIGINAL)) %>%
        mutate(INVALID_COORD_REASON = "NO_VALID_TRANSFORMATION_IN_COUNTRY") %>% 
        select(-LONGITUDE, -LATITUDE)
    
    # Step 7 - Summary logs
    n_total_coords <- sum(!is.na(coord_fix_df$LONGITUDE_ORIGINAL) & !is.na(coord_fix_df$LATITUDE_ORIGINAL))
    n_kept_original <- sum(coord_fix_df$COORD_FIX_METHOD == "ORIGINAL", na.rm = TRUE)
    n_corrected <- sum(coord_fix_df$COORD_IS_VALID & coord_fix_df$COORD_FIX_METHOD != "ORIGINAL", na.rm = TRUE)
    n_invalid <- nrow(invalid_coords)
    
    log_msg(glue("Coordinate quality check over {n_total_coords} FOSAs: original valid={n_kept_original}, corrected={n_corrected}, invalid={n_invalid}."))
    if (n_corrected > 0) {
        log_msg(glue("Applied coordinate correction algorithm to {n_corrected} FOSAs (swap/sign/decimal left-to-right, k<=2)."), "warning")
    }
    if (n_invalid > 0) {
        log_msg(glue("{n_invalid} FOSAs remain invalid after correction attempts. LONGITUDE/LATITUDE set to NA."), "warning")
    }
} else {
    invalid_coords <- c()
}

head(pyramid_data_coords, 3)

### Create formatted SNT template (.csv)

In [ ]:
# Create pyramid template
adm1_id_col <- gsub("_NAME", "_ID", ADMIN_1)
adm2_id_col <- gsub("_NAME", "_ID", ADMIN_2)

pyramid_data_template <- pyramid_data_coords %>%
    select(
        ADM1_NAME=!!sym(ADMIN_1),
        ADM1_ID=!!sym(adm1_id_col),
        ADM2_NAME=!!sym(ADMIN_2),
        ADM2_ID=!!sym(adm2_id_col)
    ) %>%
    distinct() %>%
    arrange(ADM1_NAME, ADM2_NAME)

head(pyramid_data_template, 3)

### Output data

In [ ]:
FORMATTED_DATA_PATH <- file.path(snt_paths$DATA_PATH, "dhis2", "extracts_formatted")

# write pyramid 
write_parquet(pyramid_data_coords, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_pyramid.parquet")))
write.csv(pyramid_data_coords, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_pyramid.csv")), row.names = FALSE)
log_msg(glue("Pyramid data saved under: {file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, '_pyramid.parquet'))}"))

# Write template
write.csv(pyramid_data_template, file.path(snt_paths$UPLOADS_PATH, paste0(COUNTRY_CODE, "_adm_template.csv")), row.names = FALSE)
log_msg(glue("SNT Administrative template created: {file.path(snt_paths$UPLOADS_PATH, paste0(COUNTRY_CODE, '_adm_template.csv'))}"))

# Write invalid coordinates report when needed
if (length(invalid_coords) > 0) {
    invalid_coords_report_path <- file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_pyramid_invalid_coordinates.csv"))
    write.csv(invalid_coords, invalid_coords_report_path, row.names = FALSE)
    log_msg(glue("Invalid coordinates report saved under: {invalid_coords_report_path}"), "warning")
}

### Data Summary 

In [ ]:
# Data summary
print(summary(pyramid_data_coords))